# v39 — Clause ablation pilot: does a confident wrong answer survive deleting the clause it ignored? (Google Colab)

Recitation errors on GSM-Symbolic P2 ignore a clause that P2 added, the same way in every sample, so voting and the PRM cannot see them. v39 tests a detector that needs no verifier and no gold answer: delete one sentence at a time and re-solve greedily. If deleting an informative sentence leaves the answer unchanged, the model was not using it. The detector is `clause_ablation.py`; the rules are frozen in `pretest_v39.py` and `V39_PLAN.md`.

131 stored rows (`pretest_data/v39_ablation.json`, seed 39): **37 P2 rows the model gets confidently wrong**, **74 it gets confidently right**, and 20 GSM-Plus distraction rows. 691 greedy generations of Qwen2.5-Math-7B-Instruct (4-bit), no verifier.

**Use this notebook if Kaggle is not available** (or to finish a Kaggle run): it works through all rows on one T4, about 3–4 hours. **The result file lives on Google Drive** (`MyDrive/MAS_SHT_v39/pretest_v39.json`) and is saved after every row; if the session ends, **Run all** again and it continues. Rows already finished in Kaggle's `pretest_v39_gpu0.json` / `pretest_v39_gpu1.json`, uploaded to the same Drive folder under their own names, are skipped, and Cell 4 reads all of them together.

1. Runtime → Change runtime type → **T4 GPU**. 2. Secrets: `HF_API_KEY` (Notebook access on). 3. Runtime → **Run all**.

In [ ]:
# === Cell 1 — Colab setup ===========================================================
import base64, os, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = '/content/drive/MyDrive/MAS_SHT_v39'
os.makedirs(OUT_DIR, exist_ok=True)

# The stack of the Kaggle runs. Colab's own torch is kept.
DEPS = ['transformers>=4.44.0,<4.50', 'accelerate', 'bitsandbytes', 'openai>=1.40.0',
        'python-dotenv', 'tabulate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + DEPS)
print('Deps installed.')

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

_hf = _secret('HF_API_KEY')
if _hf:
    for _k in ('HF_API_KEY', 'HUGGINGFACE_TOKEN', 'HF_TOKEN'):
        os.environ[_k] = _hf
    print('HF token loaded.')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU: Runtime -> Change runtime type -> T4 GPU, then run again.')
_p = torch.cuda.get_device_properties(0)
print(f'cuda:0 {_p.name}  sm_{_p.major}{_p.minor}  {_p.total_memory / 2**30:.0f} GB')
if (_p.major, _p.minor) < (7, 5):
    raise RuntimeError(f'{_p.name} is older than sm_75; 4-bit NF4 needs a T4 or newer.')

# The token never enters a URL (git would store it in .git/config); a one-off
# http.extraHeader is not persisted. The repository is public: no token is needed.
REPO_DIR = '/content/llm_thesis'
BRANCH   = 'main'
REPO_URL = 'https://github.com/marios4371/llm_thesis.git'
_tok = _secret('GITHUB_TOKEN')
_auth, _hide = [], []
if _tok:
    _hdr = base64.b64encode(f'x-access-token:{_tok}'.encode()).decode()
    _auth, _hide = ['-c', f'http.extraHeader=Authorization: Basic {_hdr}'], [_tok, _hdr]

def _git(*args):
    _r = subprocess.run(['git', *_auth, *args], capture_output=True, text=True)
    if _r.returncode:
        _err = _r.stderr
        for _s in _hide:
            _err = _err.replace(_s, '***')
        raise RuntimeError(f"git {' '.join(args)} failed:\n" + _err)
    return _r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git('clone', REPO_URL, REPO_DIR)
_git('-C', REPO_DIR, 'remote', 'set-url', 'origin', REPO_URL)
_git('-C', REPO_DIR, 'fetch', 'origin', BRANCH)
_git('-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}')
os.chdir(REPO_DIR)
print('code at', _git('log', '-1', '--format=%h %s'))
OUT = f'{OUT_DIR}/pretest_v39.json'
print('results go to', OUT)

In [ ]:
# === Cell 2 — offline tests (the detector and the manifest against their frozen sha256) ==========
import subprocess, sys
_t = subprocess.run([sys.executable, 'test_v39.py'], capture_output=True, text=True)
print('test_v39.py ->', _t.stdout.strip().splitlines()[-1] if _t.stdout.strip() else _t.stderr[-2000:])
if _t.returncode != 0:
    print(_t.stdout[-5000:], _t.stderr[-3000:])
    raise RuntimeError('test_v39.py failed: do NOT start the GPU run')

In [ ]:
# === Cell 3 — the run (resumes from the files on Drive) =================================
import subprocess, sys
CMD = [sys.executable, '-u', 'pretest_v39.py', '--out', OUT, '--max-hours', '11.0']
_p = subprocess.Popen(CMD, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for _line in _p.stdout:
    print(_line, end='')
_p.wait()
print('exit code', _p.returncode)

In [ ]:
# === Cell 4 — the pre-registered reading (no GPU); reads every pretest_v39*.json in the Drive folder ====
!python -u pretest_v39.py --summary-only --out "$OUT"